# Week 2 Day 2 Hard Exercise: SDR to closer on Resend, with replies and Telegram

Goes past the SMTP version:
- **Sending**: Resend API instead of SMTP.
- **Replies**: the agent reads prospect replies from your Gmail inbox over IMAP and continues the conversation.
- **Closing**: the agent works each deal through outreach, objections, and a close. It marks deals `won` or `lost`.
- **Telegram**: you get a message for every outbound email, every reply, and every deal won or needing a human.

**Orchestration choice: LLM with tools.** Each reply needs judgement: answer a question, handle an objection, propose a call, escalate, or close. An LLM with tools makes that call and each action stays a visible tool call. Python owns the loop, the stored deal state, and the guardrails (touch limit, stage checks, safe-to-self sending). Handoffs are not used, as requested.

## How it fits together

1. `start_deal(...)` writes the first email with an outreach agent and sends it through Resend.
2. Every email carries `[ref:<deal_id>]` in the subject. Replies keep that subject, so the agent can match them to a deal.
3. `run_cycle()` reads unread Gmail replies, finds the deal, and asks the **closer agent** what to do next.
4. The closer agent uses tools: `send_reply`, `set_stage`, `notify_human`.

## Setup notes

- Resend's default sender `onboarding@resend.dev` only delivers to your own account email. That matches `SEND_TO_SELF = True`. For real prospects, verify a domain in Resend and set `RESEND_FROM` in `.env`.
- Resend cannot receive mail on the free setup. Replies go to your Gmail via `Reply-To`, and IMAP reads them. Gmail needs IMAP enabled and an app password (you already have `EMAIL_APP_PASSWORD`).
- Telegram: message your bot once so it can message you back, then `TELEGRAM_CHAT_ID` is the chat to send to.

To test a conversation: start a deal, then reply to the email in Gmail as if you were the prospect, then run `run_cycle()`.


## Part 1: Setup


In [ ]:
import os
import re
import json
import time
import asyncio
import imaplib
import email
import requests
from email.policy import default as default_policy
from email.header import make_header, decode_header
from email.utils import parseaddr

from dotenv import load_dotenv
from pydantic import BaseModel
from agents import Agent, Runner, trace, function_tool

load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

RESEND_API_KEY = os.getenv("RESEND_API_KEY")
RESEND_FROM = os.getenv("RESEND_FROM", "onboarding@resend.dev")
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")  # your Gmail: replies land here
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

SEND_TO_SELF = True   # while True, every outbound email goes to SELF_RECIPIENT
# Resend's onboarding sender only delivers to the Resend account owner until a domain is verified
SELF_RECIPIENT = os.getenv("EMAIL_ADDRESS2")
MAX_TOUCHES = 6       # guardrail: stop emailing a prospect after this many messages
DEALS_FILE = "deals.json"
VALID_STAGES = {"outreach", "negotiating", "won", "lost"}

print("Resend set up:", bool(RESEND_API_KEY))
print("Telegram set up:", bool(TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID))
print("Gmail IMAP set up:", bool(EMAIL_ADDRESS and EMAIL_APP_PASSWORD))


## Part 2: Deal store, Telegram, and Resend


In [ ]:
class Email(BaseModel):
    subject: str
    text_body: str
    html_body: str


REF_RE = re.compile(r"\[ref:([\w-]+)\]")


def extract_ref(subject):
    m = REF_RE.search(subject or "")
    return m.group(1) if m else None


# Quick self-check of the reply matcher
assert extract_ref("Re: Quick question [ref:acme-1700000000]") == "acme-1700000000"
assert extract_ref("Hello there") is None


def load_deals():
    if os.path.exists(DEALS_FILE):
        with open(DEALS_FILE, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_deals(deals):
    with open(DEALS_FILE, "w", encoding="utf-8") as f:
        json.dump(deals, f, indent=2)


def notify(text):
    """Send a Telegram message to you. Falls back to print if Telegram is not set up."""
    if not (TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID):
        print("[telegram not set]", text)
        return
    r = requests.post(
        f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage",
        json={"chat_id": TELEGRAM_CHAT_ID, "text": text[:4000]},
        timeout=15,
    )
    r.raise_for_status()


def send_resend(deal_id, to_addr, subject, text_body, html_body):
    """Send one email through Resend. The [ref:...] tag is added so replies can be matched."""
    clean_subject = REF_RE.sub("", subject).strip()
    payload = {
        "from": RESEND_FROM,
        "to": [SELF_RECIPIENT if SEND_TO_SELF else to_addr],
        "subject": f"{clean_subject} [ref:{deal_id}]",
        "text": text_body,
        "html": html_body,
        "reply_to": EMAIL_ADDRESS,
    }
    r = requests.post(
        "https://api.resend.com/emails",
        headers={"Authorization": f"Bearer {RESEND_API_KEY}"},
        json=payload,
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["id"]


def send_and_log(deal_id, subject, text_body, html_body):
    """Send via Resend, record the touch in the deal, and tell Telegram."""
    deals = load_deals()
    deal = deals[deal_id]
    if deal["touches"] >= MAX_TOUCHES:
        return f"Blocked: touch limit of {MAX_TOUCHES} reached. Set the stage to lost instead."
    send_resend(deal_id, deal["prospect"]["email"], subject, text_body, html_body)
    deal["touches"] += 1
    deal["history"].append({"from": "us", "subject": subject, "text": text_body})
    save_deals(deals)
    notify(f"Sent to {deal['prospect']['company']} ({deal_id}): {subject}")
    return "sent"


## Part 3: Reading replies from Gmail


In [ ]:
def fetch_replies():
    """Unread Gmail messages whose subject carries a [ref:...] tag. Our own sends are skipped."""
    imap = imaplib.IMAP4_SSL("imap.gmail.com", timeout=30)  # shortcut: no retry on timeout, rerun the cell
    imap.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
    imap.select("INBOX")
    _, data = imap.uid("search", None, "UNSEEN", "SUBJECT", "ref:")  # only our threads, not the whole inbox

    replies = []
    for num in data[0].split():
        _, msg_data = imap.uid("fetch", num, "(RFC822)")
        msg = email.message_from_bytes(msg_data[0][1], policy=default_policy)
        subject = str(make_header(decode_header(str(msg["Subject"] or ""))))
        deal_id = extract_ref(subject)
        sender = parseaddr(str(msg["From"]))[1]
        if sender == RESEND_FROM:  # our own outbound copy, not a reply: mark it read and skip
            imap.uid("store", num, "+FLAGS", "\\Seen")
            continue
        if not deal_id:
            continue

        body = msg.get_body(preferencelist=("plain",))
        text = body.get_content() if body else ""
        # Drop the quoted history below the reply
        text = re.split(r"\n(?=On .+wrote:|>)", text)[0].strip()

        replies.append({"uid": num, "deal_id": deal_id, "from": sender, "subject": subject, "text": text})
    imap.logout()
    return replies


## Part 4: Tools for the closer agent

The LLM chooses which tool to call. Each tool checks its own guardrails.


In [ ]:
@function_tool
def send_reply(deal_id: str, subject: str, text_body: str, html_body: str) -> str:
    """Send the next email in a deal's thread. Use subject 'Re: <original subject>'. Keep the body under 120 words."""
    return send_and_log(deal_id, subject, text_body, html_body)


@function_tool
def set_stage(deal_id: str, stage: str) -> str:
    """Set a deal's stage: negotiating, won or lost. Use won only when the prospect has clearly agreed to buy."""
    if stage not in VALID_STAGES:
        return f"Invalid stage. Use one of {sorted(VALID_STAGES)}."
    deals = load_deals()
    deals[deal_id]["stage"] = stage
    save_deals(deals)
    if stage == "won":
        notify(f"DEAL WON: {deals[deal_id]['prospect']['company']} ({deal_id}). Send the contract.")
    if stage == "lost":
        notify(f"Deal lost: {deals[deal_id]['prospect']['company']} ({deal_id}).")
    return f"stage={stage}"


@function_tool
def notify_human(deal_id: str, message: str) -> str:
    """Message the human owner on Telegram. Use for pricing, legal or anything you cannot answer confidently."""
    notify(f"[{deal_id}] {message}")
    return "notified"


## Part 5: The agents

- **Outreach writer**: writes the first cold email.
- **Closer**: the orchestrating LLM. Given a deal and the latest reply, it picks the next step.


In [ ]:
outreach_writer = Agent(
    name="Outreach Writer",
    model=MODEL_NAME,
    output_type=Email,
    instructions="""
You write the first cold email for ComplAI, a SaaS tool for SOC2 compliance and audit preparation, powered by AI.
Address the prospect by first name, mention their company once, and propose a short call.
Under 120 words. Professional and warm. Return subject, text_body and html_body.
""",
)

closer = Agent(
    name="SDR Closer",
    model=MODEL_NAME,
    tools=[send_reply, set_stage, notify_human],
    instructions="""
You run the sales conversation for ComplAI. You get one deal (JSON with deal_id, prospect, stage, history) and the prospect's latest reply.
Take exactly one next step:
- A question or interest: answer it briefly and propose a 20-minute call. Use send_reply.
- An objection: acknowledge it, answer in 2-3 sentences, and propose a next step. Use send_reply.
- Pricing, contracts, or anything you cannot confirm: do not promise numbers. Use notify_human with a short summary.
- A clear yes to buying: use set_stage with "won", then notify_human.
- A clear no, or they asked to stop: use set_stage with "lost".
Use deal_id exactly as given. Emails are under 120 words. Use subject 'Re: <original subject>'.
Finish with a one-line summary of what you did.
""",
)


## Part 6: The loop

`start_deal` opens a conversation. `handle_reply` lets the closer agent respond to one reply. `run_cycle` checks the inbox once. `run_loop` repeats `run_cycle` on a timer.


In [ ]:
async def start_deal(prospect):
    deal_id = f"{re.sub(r'[^a-z0-9]+', '-', prospect['company'].lower()).strip('-')}-{int(time.time())}"
    draft = (await Runner.run(outreach_writer, json.dumps(prospect))).final_output

    deals = load_deals()
    deals[deal_id] = {
        "deal_id": deal_id,
        "prospect": prospect,
        "stage": "outreach",
        "touches": 0,
        "history": [],
    }
    save_deals(deals)
    print(send_and_log(deal_id, draft.subject, draft.text_body, draft.html_body))
    return deal_id


async def handle_reply(reply):
    deals = load_deals()
    deal = deals.get(reply["deal_id"])
    if not deal or deal["stage"] in ("won", "lost"):
        print("Ignoring reply for closed or unknown deal", reply["deal_id"])
        return
    if any(h["text"] == reply["text"] for h in deal["history"]):  # already answered; mark_seen failed last time
        return

    deal["history"].append({"from": "prospect", "subject": reply["subject"], "text": reply["text"]})
    save_deals(deals)
    notify(f"Reply from {deal['prospect']['company']}:\n{reply['text'][:500]}")

    prompt = f"Deal:\n{json.dumps(deal, indent=2)}\n\nLatest prospect reply:\n{reply['text']}"
    with trace(f"closer {reply['deal_id']}"):
        result = await Runner.run(closer, prompt)
    print(result.final_output)


def mark_seen(uid):
    """Mark a reply read only after the closer has handled it, so a crash never drops it."""
    imap = imaplib.IMAP4_SSL("imap.gmail.com", timeout=30)
    imap.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
    imap.select("INBOX")
    imap.uid("store", uid, "+FLAGS", "\\Seen")
    imap.logout()


async def run_cycle():
    for reply in fetch_replies():
        await handle_reply(reply)
        mark_seen(reply["uid"])
    print("Inbox checked.")


async def run_loop(interval_s=60, cycles=10):
    for _ in range(cycles):
        try:
            await run_cycle()
        except Exception as e:  # Gmail IMAP times out now and then; retry on the next tick
            print("cycle failed, retrying next tick:", repr(e), flush=True)
        await asyncio.sleep(interval_s)


## Part 7: Run it

1. Check Telegram works.
2. Start a deal. You receive the email in Gmail.
3. Reply to it from Gmail, then run `run_cycle()`. The closer answers and you get a Telegram message.
4. Repeat until the deal is won or lost, or run `run_loop()` to keep checking.


In [ ]:
notify("ComplAI SDR closer is online.")


In [ ]:
prospect = {
    "first_name": "Ada",
    "company": "Lovelace Labs",
    "role": "Head of Compliance",
    "email": "ada@example.com",  # ignored while SEND_TO_SELF is True
}

with trace("start deal"):
    deal_id = await start_deal(prospect)
print("Deal started:", deal_id)


In [ ]:
await run_cycle()


In [ ]:
# Optional: keep checking every 60 seconds for 30 checks (about 30 minutes)
# await run_loop(interval_s=60, cycles=30)
